# Drone Chase Tag: tarama (birkaç koşu arka arkaya)

Bir ayarın farklı değerlerini karşılaştırmak için: koşular sırayla çalışır, sonunda hepsi tek grafikte
karşılaştırılır ve tek zip olarak iner.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU** (yoksa L4) seç.
2. Hücreleri sırayla çalıştır. Oturum koparsa aynı hücreleri tekrar çalıştır: biten koşular atlanır, yarım kalan
   checkpoint'ten devam eder.
3. En sondaki **Arşiv** hücresi bütün koşuları zip olarak bilgisayarına indirir.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content")                        # hücre nerede çalışırsa çalışsın, önce sabit bir yere geç
REPO = "https://github.com/HasaNSaygili/DroneChaseTag.git"
if os.path.isdir("/content/proje/.git"):    # daha önce indirildiyse silmeden güncelle
    !git -C /content/proje fetch -q origin && git -C /content/proje reset -q --hard origin/main
else:
    !git clone -q {REPO} /content/proje
assert os.path.isfile("/content/proje/scripts/train.py"), "Kod indirilemedi: bu hücrenin çıktısını sohbete yapıştır"
os.chdir("/content/proje")
!git log --oneline -1
!pip install -q "crazyflow[gpu]==0.4.0" -e .

## Tarama ayarları
`SWEEPS` listesinde bütün taramalar duruyor; `SWEEP_NAME`'e çalıştırmak istediğinin adını yaz.
Her koşu: (script, ayarlar). Koşu adları `experiments/LOG.md` ile aynı.

In [ ]:
FAZ1 = '"n_obst=(8,12)" opp_crash_reward=0.0'
LONG = " ent_coef=0.001 total_steps=1500000000"   # engelli arenada 500 M yetmiyor (bkz. sweep-obs-chaser-faz1)
CHASER = " learner=chaser opponent=open gamma=0.995"
EVADER = " learner=evader opponent=scripted_chaser gamma=0.998"
SWEEPS = {
    # 1) Entropi aralığı: engelli arena (faz 1, 8–12 engel), SIFIRDAN kovalayan vs OPEN kaçanı, sınırlı ortalama.
    #    Değerler logaritmik ölçekte eşit aralıklı: her biri bir öncekinin 2 katı (entropinin etkisi oranla değişir).
    #    Bakılacaklar: öğrenme hızı ve son ebeleme oranı, std'nin seyri, gaz pedalı (kırpılan oranı, düz bölge).
    "sweep-ent-chaser-faz1": {
        "ppo-chaser-faz1-ent0":     ("train", FAZ1 + " ent_coef=0.0"),
        "ppo-chaser-faz1-ent0.001": ("train", FAZ1 + " ent_coef=0.001"),
        "ppo-chaser-faz1-ent0.002": ("train", FAZ1 + " ent_coef=0.002"),
        "ppo-chaser-faz1-ent0.004": ("train", FAZ1 + " ent_coef=0.004"),
        "ppo-chaser-faz1-ent0.008": ("train", FAZ1 + " ent_coef=0.008"),
    },
    # 2) Darboğaz görüş/hafıza mı? Entropi taramasında kovalayan rakibi kaybedince arayamıyordu (ebeleme ~%30).
    #    A) tam gözlem: rakip hep görünür. Yüksek çıkarsa sorun bilgi eksikliği (→ hafıza), düşükse yol bulma/ödül.
    #    B) maskeli ama 3 kat uzun (1,5 milyar adım): zamanla kendi arama alışkanlığını öğrenebiliyor mu?
    "sweep-obs-chaser-faz1": {
        "ppo-chaser-faz1-fullobs": ("train", FAZ1 + " ent_coef=0.001 full_obs=True"),
        "ppo-chaser-faz1-long":    ("train", FAZ1 + " ent_coef=0.001 total_steps=1500000000"),
    },
    # 3) Engelli arenada temel: elle yazılmış rakiplere karşı, 1,5 milyar adım, birden çok seed.
    #    Kaçan vs elle yazılmış kovalayan (self-play'in kaçan_0'ı; saklanmayı öğreniyor mu?) 3 seed;
    #    kovalayan vs OPEN için ek 2 seed (seed 0 = ppo-chaser-faz1-long). Seed'ler sonucun şans olmadığını gösterir.
    "sweep-base-faz1": {
        "ppo-evader-faz1-long-s0": ("train", FAZ1 + LONG + EVADER + " seed=0"),
        "ppo-chaser-faz1-long-s1": ("train", FAZ1 + LONG + CHASER + " seed=1"),
        "ppo-evader-faz1-long-s1": ("train", FAZ1 + LONG + EVADER + " seed=1"),
        "ppo-chaser-faz1-long-s2": ("train", FAZ1 + LONG + CHASER + " seed=2"),
        "ppo-evader-faz1-long-s2": ("train", FAZ1 + LONG + EVADER + " seed=2"),
    },
}
SWEEP_NAME = "sweep-base-faz1"
OUT = "/content/drive/MyDrive/DroneChaseTag/results"
RUNS = SWEEPS[SWEEP_NAME]
for name, (script, s) in RUNS.items():
    print(f"{name:28s} | {script} | {s}")

In [ ]:
import os
os.chdir("/content/proje")
for name, (script, s) in RUNS.items():
    print(f"\n################ {name} ################", flush=True)
    !python -W ignore scripts/{script}.py --name {name} --out {OUT} {('--set ' + s) if s else ''} 2>&1 | grep -v -E "warp|spec|fromstring"

## Değerlendirme
Her koşu hiç görülmemiş arenalarda, gürültüsüz oynatılır; birer maç videoya çekilir (Drive'da `videos/`).

In [ ]:
import os
os.chdir("/content/proje")
for name, (script, s) in RUNS.items():
    if script == "train":
        print(f"\n######## {name}", flush=True)
        !MUJOCO_GL=egl python -W ignore scripts/watch_policy.py --run {OUT}/{name} --games 1 --eval-worlds 1024 --out /content/drive/MyDrive/DroneChaseTag/videos 2>&1 | grep -v -E "warp|spec|fromstring"

## Karşılaştırma
Üst sıra: öğrenme (öğrenenin kazanma oranı, öğrenenin çarpma oranı, maç süresi).
Alt sıra: keşif ve gaz pedalı (std, denenen eylemlerin kırpılan oranı, tanh'ın düz bölgesindeki eksen oranı).

In [ ]:
import glob, json, numpy as np, matplotlib.pyplot as plt
K = 5
smooth = lambda y: np.convolve(y, np.ones(K) / K, 'valid') if len(y) > K else np.asarray(y)
fig, ax = plt.subplots(2, 3, figsize=(17, 8))
panels = [('kazanma (öğrenen)', 0, 0), ('çarpma (öğrenen)', 0, 1), ('maç süresi (s)', 0, 2),
          ('std (keşif)', 1, 0), ('kırpılan eylem oranı', 1, 1), ('tanh düz bölge oranı', 1, 2)]
summary = []
for name in RUNS:
    d = f"{OUT}/{name}"
    if not os.path.exists(f"{d}/metrics.jsonl"):
        print("yok:", name); continue
    cfg = json.load(open(f"{d}/config.json"))
    me = 0 if cfg['learner'] == 'chaser' else 1
    rows = [json.loads(l) for l in open(f"{d}/metrics.jsonl")]
    x = np.array([r['steps'] for r in rows]) / 1e6
    C = np.array([r['outcome_counts'] for r in rows]); tot = C[:, 1:].sum(1).clip(1)
    win = (C[:, 1] if me == 0 else C[:, 2]) / tot          # kovalayan: ebeleme, kaçan: süre doldu
    crash = (C[:, 4] if me == 0 else C[:, 3]) / tot        # öğrenenin kendi çarpması
    series = [win, crash, np.array([r['ep_len'] for r in rows]) / 50, [r['std'] for r in rows],
              [r.get('act_clip_frac', np.nan) for r in rows], [r.get('pre_sat_frac', np.nan) for r in rows]]
    for (title, i, j), y in zip(panels, series):
        ys = smooth(np.asarray(y, float)); xs = x[(len(x) - len(ys)) // 2:][:len(ys)]
        ax[i, j].plot(xs, ys, label=name.replace('ppo-', ''))
        ax[i, j].set_title(title)
    ev = sorted(glob.glob(f"{d}/eval_*.json"))
    e = json.load(open(ev[-1]))['outcomes_pct'] if ev else {}
    summary.append((name, e, rows[-1]))
for a in ax.flat: a.set_xlabel('M adım')
ax[0, 0].legend(fontsize=9)
fig.suptitle(SWEEP_NAME); fig.tight_layout()
fig.savefig(f"{OUT}/{SWEEP_NAME}.png", dpi=120, bbox_inches='tight')
plt.show()
print("\nÖZET (bunu sohbete yapıştır)")
for name, e, last in summary:
    print(f"{name:28s} | değerlendirme: " + " ".join(f"{k} %{v:.1f}" for k, v in e.items()) +
          f" | son tur: std {last['std']:.2f} kırpılan %{100 * last.get('act_clip_frac', np.nan):.0f} "
          f"düz %{100 * last.get('pre_sat_frac', np.nan):.0f} süre {last['ep_len'] / 50:.1f} s")

## Arşiv: bütün koşuları bilgisayarına indir
Her koşunun küçük dosyaları + son politikası + karşılaştırma grafiği tek zip olarak iner.

In [ ]:
import glob, os, zipfile
from google.colab import files
zpath = f"/content/{SWEEP_NAME}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    if os.path.exists(f"{OUT}/{SWEEP_NAME}.png"):
        z.write(f"{OUT}/{SWEEP_NAME}.png", f"{SWEEP_NAME}.png")
    for name in RUNS:
        d = f"{OUT}/{name}"
        keep = [p for p in glob.glob(f"{d}/*") if not p.endswith("checkpoint.pkl") and not os.path.basename(p).startswith("params_")]
        keep += sorted(glob.glob(f"{d}/params_*.pkl"))[-1:]
        for p in keep:
            z.write(p, f"{name}/{os.path.basename(p)}")
print(zpath)
files.download(zpath)